# 🧮 GLM-OCR v3.1 — Handwritten Math → LaTeX

**Fine-tuned OCR model for handwritten mathematics**, built by fine-tuning
[zai-org/GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) with LoRA on **~4,672
annotated pages** of handwritten undergraduate mathematics, warm-started from an
earlier v3 adapter.

The model reads a photo or scan of a handwritten math page and produces a
**complete, compilable LaTeX document** — ignoring printed text, student names,
page numbers, and cancelled rough work.

---

## 📋 How to use this notebook (3 steps)

1. **Enable the GPU**: menu **Runtime → Change runtime type → T4 GPU → Save**
2. **Run setup**: menu **Runtime → Run all** (or run cells top-to-bottom).  
   First-time setup takes **≈ 5–10 minutes** (installs LaTeX + downloads the model).
3. **Upload your own page** at **Step 6** and see both models side-by-side.

A set of sample pages is included so you can see results without uploading anything.

| What you get per page | |
|---|---|
| ⚪ **Base GLM-OCR** output | what the original, un-trained model produces |
| 🔵 **v3.1 Fine-tuned** output | what the fine-tuned model produces |
| 📄 Compiled PDFs | rendered side-by-side with the original image |
| ⏱ Timing | seconds per page for each model |

| Metric (700-page held-out benchmark) | Base GLM-OCR | v3.1 Fine-tuned | Improvement |
|---|---|---|---|
| CER (↓ better) | 0.515 | **0.397** | −23% |
| PDF Compile Rate (↑ better) | ~0% | **88.9%** | +89 pts |
| Math-F1 (↑ better) | 0.703 | **0.817** | — |
| Structural Match (↑ better) | 0% | **91.4%** | — |

> ⚠️ On the free Colab T4 GPU, each page takes roughly **1–3 minutes per model**.

## Step 1 — Check the GPU

This must print a GPU name (e.g. *Tesla T4*). If it fails:  
**Runtime → Change runtime type → T4 GPU → Save**, then re-run this cell.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "❌ No GPU detected!\n"
    "Fix: menu 'Runtime' → 'Change runtime type' → Hardware accelerator: 'T4 GPU' → Save."
)
gpu = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
# bf16 needs compute capability >= 8.0 (A100/L4); T4 (7.5) uses fp16
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
print(f"✅ GPU: {gpu}  |  dtype for inference: {DTYPE}")
!nvidia-smi -L

## Step 2 — Install dependencies (≈ 4–6 min)

Installs Python libraries and a LaTeX compiler (used to render the model's output as a PDF).
The LaTeX install is the slow part — please be patient.

In [ ]:
%%time
# Python libraries
!pip install -q -U transformers peft accelerate torchao>=0.16.0
!pip install -q pillow pymupdf gdown pandas

# LaTeX compiler (to render .tex output into PDFs)
print("Installing LaTeX (this is the slow part, ~3–4 min)...")
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y texlive-latex-base texlive-fonts-recommended texlive-latex-extra > /dev/null 2>&1

import shutil
print("✅ pdflatex:", shutil.which("pdflatex") or "NOT FOUND (PDFs disabled, LaTeX text still works)")

## Step 3 — Download the fine-tuned adapter (~83 MB)

Downloads the LoRA adapter (v3.1 fine-tuned weights) from Google Drive.
The file ID is already filled in — **just run the cell**.

<details><summary>⚠️ If the download fails (click to expand)</summary>

Google Drive occasionally rate-limits downloads. If that happens:
1. Download the zip manually from the share link you were sent.
2. In Colab, click the **📁 folder icon** (left sidebar) → **upload** the zip
   to `/content/`, naming it `glm_ocr_v31_package.zip`.
3. Re-run this cell — it will detect the uploaded file and skip the download.
</details>

In [ ]:
import os, zipfile
from pathlib import Path

# ── Filled in by the notebook owner before sharing ────────────────────────────────
ADAPTER_DRIVE_FILE_ID = '1UfDBcRyNpKlVVET-IpBpt8GFbLzapk10'
# ────────────────────────────────────────────────────────────────

PKG_ZIP     = Path("/content/glm_ocr_v31_package.zip")
PKG_DIR     = Path("/content/glm_ocr_package")
ADAPTER_DIR = PKG_DIR / "adapter"
SAMPLES_DIR = PKG_DIR / "sample_images"

if not PKG_ZIP.exists():
    import gdown
    gdown.download(id=ADAPTER_DRIVE_FILE_ID, output=str(PKG_ZIP), quiet=False)

if not ADAPTER_DIR.exists():
    with zipfile.ZipFile(PKG_ZIP) as z:
        z.extractall(PKG_DIR)

assert (ADAPTER_DIR / "adapter_config.json").exists(), (
    f"adapter_config.json not found — zip layout unexpected. "
    f"Contents: {list(PKG_DIR.rglob('*'))[:10]}"
)

samples = sorted(SAMPLES_DIR.glob("*.png")) if SAMPLES_DIR.exists() else []
print(f"✅ Adapter ready: {ADAPTER_DIR}")
print(f"✅ Sample pages included: {[p.name for p in samples]}")

## Step 4 — Load the model (≈ 3–5 min, one-time)

Downloads the base **GLM-OCR** model from Hugging Face and attaches the
fine-tuned LoRA adapter **without merging it**. This means *one* model in GPU
memory can act as **both** models:

- adapter **enabled** → fine-tuned **v3.1** (warm-started from v3, trained on 4,672 pages, LoRA r=32)
- adapter **disabled** → original **base** GLM-OCR

Identical prompt, resolution, and decoding settings for both — a fair comparison.

In [ ]:
%%time
import os
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"❌ Adapter not found at {ADAPTER_DIR}. Please re-run Step 3.")

MODEL_NAME = "zai-org/GLM-OCR"

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Loading base model (this takes a few minutes)...")
try:
    base = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME, trust_remote_code=True, dtype=DTYPE)
except TypeError:
    base = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME, trust_remote_code=True, torch_dtype=DTYPE)

print("Attaching v3.1 adapter...")
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))   # NOT merged — toggleable
model.to("cuda")
model.eval()

# Match the image resolution used during fine-tuning (1536 image tokens)
ip = processor.image_processor
patch = getattr(ip, "patch_size", 14)
merge = getattr(ip, "merge_size", 2)
ip.size["longest_edge"] = int(1536 * patch * patch * merge * merge)

print("✅ Model loaded successfully!")

## Step 5 — Helper functions

Defines `run_page(image_path)` which runs **both models** on a page,
compiles each output to PDF, shows everything side-by-side, and displays
**copyable LaTeX boxes** below the figure:

- ⚪ **Base GLM-OCR** — always shows a copy box (base rarely compiles, so the raw LaTeX is the only output)
- 🔵 **v3.1 Fine-tuned** — always shows a copy box; if the PDF compiled you can still grab the source; if it failed you can paste into Overleaf to fix it

In [ ]:
import io, re, shutil, subprocess, tempfile, time, textwrap, html as _html
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, HTML

RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(exist_ok=True)

USER_PROMPT = (
    "OCR this handwritten math page. Convert ONLY the handwritten mathematical "
    "content into a complete, compilable LaTeX document. Ignore printed text, "
    "student info, page numbers, cancelled work and rough work. Output only LaTeX."
)

_copy_uid = 0  # unique id counter so multiple pages never share button ids


def _latex_copy_box(label: str, latex: str, color: str = "#555") -> str:
    """Scrollable textarea + Copy button + Open in Overleaf button."""
    global _copy_uid
    _copy_uid += 1
    uid = f"ltx_{_copy_uid}"
    btn = f"btn_{_copy_uid}"
    esc = _html.escape(latex or "(empty)")

    copy_js = (
        f"var t=document.getElementById('{uid}');"
        f"t.select();t.setSelectionRange(0,99999);"
        f"try{{navigator.clipboard.writeText(t.value).then(function(){{"
        f"  document.getElementById('{btn}').innerText='✅ Copied!';"
        f"  setTimeout(function(){{document.getElementById('{btn}').innerText='📋 Copy LaTeX';}},1800);"
        f"}})}}catch(e){{"
        f"  document.execCommand('copy');"
        f"  document.getElementById('{btn}').innerText='✅ Copied!';"
        f"  setTimeout(function(){{document.getElementById('{btn}').innerText='📋 Copy LaTeX';}},1800);"
        f"}}"
    )

    overleaf_js = (
        f"var t=document.getElementById('{uid}');"
        f"var f=document.createElement('form');"
        f"f.method='POST';f.action='https://www.overleaf.com/docs';f.target='_blank';"
        f"var i=document.createElement('input');"
        f"i.type='hidden';i.name='snip';i.value=t.value;"
        f"f.appendChild(i);document.body.appendChild(f);f.submit();document.body.removeChild(f);"
    )

    btn_style = (
        "padding:5px 12px;border:1px solid #bbb;border-radius:5px;background:#fff;"
        "cursor:pointer;font-size:12px;font-family:sans-serif;white-space:nowrap;flex-shrink:0"
    )

    return f"""
<div style="margin:6px 0 16px 0;border:1px solid #ddd;border-radius:8px;overflow:hidden;font-family:monospace">
  <div style="background:#f5f5f5;padding:8px 12px;display:flex;justify-content:space-between;align-items:center;gap:8px">
    <span style="font-weight:600;color:{color};font-size:12.5px;flex:1">{label}</span>
    <div style="display:flex;gap:6px;flex-shrink:0">
      <button id="{btn}" onclick="{copy_js}" style="{btn_style}">📋 Copy LaTeX</button>
      <button onclick="{overleaf_js}"
        style="{btn_style};background:#4f9a44;color:#fff;border-color:#3d7a34"
        title="Opens a new Overleaf project with this LaTeX pre-loaded — ready to compile">
        <img src="https://www.overleaf.com/favicon.ico"
             style="width:12px;height:12px;vertical-align:middle;margin-right:4px"
             onerror="this.style.display='none'">Open in Overleaf ↗
      </button>
    </div>
  </div>
  <textarea id="{uid}" readonly
    style="width:100%;height:180px;border:none;padding:10px 12px;font-size:11.5px;
           line-height:1.55;resize:vertical;background:#fafafa;box-sizing:border-box;
           font-family:monospace"
  >{esc}</textarea>
</div>"""


def _generate(image: Image.Image) -> str:
    messages = [{"role": "user", "content": [{"type": "image"},
                                              {"type": "text", "text": USER_PROMPT}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.0,
        )
    plen = inputs["input_ids"].shape[1]
    return processor.tokenizer.decode(out[0][plen:], skip_special_tokens=True)


def run_ocr(image_path, version: str):
    image = Image.open(image_path).convert("RGB")
    t0 = time.time()
    if version == "base":
        with model.disable_adapter():
            latex = _generate(image)
    else:
        latex = _generate(image)
    return latex, round(time.time() - t0, 1)


def compile_latex(latex_src: str, tag: str):
    if not shutil.which("pdflatex"):
        return None, "pdflatex not installed"
    with tempfile.TemporaryDirectory() as tmp:
        tex = Path(tmp) / f"{tag}.tex"
        tex.write_text(latex_src, encoding="utf-8")
        for _ in range(2):
            try:
                r = subprocess.run(
                    ["pdflatex", "-interaction=nonstopmode", tex.name],
                    cwd=tmp, capture_output=True, text=True, timeout=90
                )
            except subprocess.TimeoutExpired:
                return None, "pdflatex timed out"
        pdf = Path(tmp) / f"{tag}.pdf"
        return (pdf.read_bytes(), r.stdout[-800:]) if pdf.exists() else (None, r.stdout[-800:])


def pdf_to_image(pdf_bytes: bytes):
    import fitz
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    pix = doc[0].get_pixmap(matrix=fitz.Matrix(2, 2))
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    doc.close()
    return img


def _plain_text_panel(latex: str):
    """Stripped version of latex for matplotlib (removes $ _ ^ to avoid render crashes)."""
    clean = latex.replace("$", "").replace("_", " ").replace("^", " ")
    return ("text", "\n".join(textwrap.wrap(clean, width=60))[:2000])


def run_page(image_path, run_base=True, run_v31=True, save=True):
    image_path = Path(image_path)
    stem = image_path.stem
    print(f"\n{'='*70}\n📄  {image_path.name}\n{'='*70}")

    results = {}

    for version, flag in [("base", run_base), ("v3.1", run_v31)]:
        if not flag:
            continue
        print(f"  ⏳ {version} running …", end=" ", flush=True)
        latex, secs = run_ocr(image_path, version)

        pdf_bytes = None
        if version == "v3.1":
            pdf_bytes, _log = compile_latex(latex, f"{stem}_v31")
            status = "PDF ✅ compiled" if pdf_bytes else "PDF ❌ failed"
        else:
            status = "(compilation skipped for base)"

        results[version] = {"latex": latex, "secs": secs, "pdf": pdf_bytes}
        print(f"done in {secs}s — {len(latex)} chars — {status}")

        if save:
            (RESULTS_DIR / f"{stem}__{version}.tex").write_text(latex, encoding="utf-8")
            if pdf_bytes:
                (RESULTS_DIR / f"{stem}__v31.pdf").write_bytes(pdf_bytes)

    # ── Matplotlib figure ─────────────────────────────────────────────────────
    panels = [("Original page", Image.open(image_path))]

    if "base" in results:
        r = results["base"]
        panels.append((f"⚪ Base GLM-OCR ({r['secs']}s)", _plain_text_panel(r["latex"])))

    if "v3.1" in results:
        r = results["v3.1"]
        title = f"🔵 v3.1 Fine-tuned ({r['secs']}s)"
        if r["pdf"]:
            panels.append((title, pdf_to_image(r["pdf"])))
        else:
            panels.append((title + " — PDF failed", _plain_text_panel(r["latex"])))

    fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 10))
    if len(panels) == 1:
        axes = [axes]
    for ax, (title, content) in zip(axes, panels):
        ax.set_title(title, fontsize=12)
        ax.axis("off")
        if isinstance(content, tuple) and content[0] == "text":
            ax.text(0.01, 0.99, content[1], transform=ax.transAxes,
                    verticalalignment="top", fontsize=8,
                    family="monospace", color="black", wrap=True, usetex=False)
        elif content is not None:
            ax.imshow(content)
        else:
            ax.text(0.5, 0.5, "(no output)", ha="center", va="center", color="red")
    plt.tight_layout()
    plt.show()

    # ── Copyable LaTeX boxes ──────────────────────────────────────────────
    copy_html = (
        "<div style='font-family:sans-serif;font-size:12px;color:#666;margin:10px 0 4px 0'>"
        "📋 <b>LaTeX source</b> — copy to clipboard or open directly in Overleaf</div>"
    )

    if "base" in results:
        r = results["base"]
        copy_html += _latex_copy_box(
            f"⚪ Base GLM-OCR · {len(r['latex'])} chars · {r['secs']}s"
            " · raw output (rarely compiles — use as reference only)",
            r["latex"],
            color="#555",
        )

    if "v3.1" in results:
        r = results["v3.1"]
        suffix = ("PDF compiled ✅ — source below for reference"
                  if r["pdf"] else
                  "PDF failed ❌ — open in Overleaf to fix")
        copy_html += _latex_copy_box(
            f"🔵 v3.1 Fine-tuned · {len(r['latex'])} chars · {r['secs']}s · {suffix}",
            r["latex"],
            color="#1a56b0",
        )

    display(HTML(copy_html))
    return results


print("✅ Helper functions ready.")

## Step 6 — Quick demo on bundled sample pages ✨

Runs **both models** on sample pages from the project's held-out test set
(the model has **never seen these pages during training**).

What to expect:
- ⚪ **Base** usually produces fragmentary, non-compiling LaTeX — the raw model
  has no sense of document structure for handwritten input.
- 🔵 **v3.1** produces a clean, compilable LaTeX document with correct math environments.

In [ ]:
if samples:
    for sample in samples[:1]:   # run 1 sample page by default (~2-3 min on T4)
        demo_results = run_page(sample)
else:
    print("No sample images found in the package — upload your own page in Step 7.")

## Step 7 — Upload your own page(s) 📤

Run the cell below — an **upload button** appears. Select one or more images
(`.png` / `.jpg`) of handwritten math pages.

Tips for best results:
- Photograph the page straight-on, evenly lit, full page in frame
- Roughly A4 proportions, handwriting clearly legible
- One page per image

In [ ]:
from google.colab import files

UPLOAD_DIR = Path("/content/my_pages")
UPLOAD_DIR.mkdir(exist_ok=True)
uploaded = files.upload()
my_pages = []
for name, data in uploaded.items():
    p = UPLOAD_DIR / name
    p.write_bytes(data)
    my_pages.append(p)
print(f"✅ {len(my_pages)} page(s) uploaded: {[p.name for p in my_pages]}")

## Step 8 — Run models on your uploaded pages

Set which models to run, then execute.  
With both models on the free T4 GPU, budget **≈ 2–5 minutes per page**.

In [ ]:
RUN_BASE = True    # ⚪ original GLM-OCR  (set False to skip and save time)
RUN_V31  = True    # 🔵 fine-tuned v3.1

import pandas as pd

rows = []
for p in sorted(UPLOAD_DIR.glob("*")):
    if p.suffix.lower() not in (".png", ".jpg", ".jpeg", ".bmp", ".webp"):
        continue
    res = run_page(p, run_base=RUN_BASE, run_v31=RUN_V31)
    row = {"page": p.name}
    for v, r in res.items():
        row[f"{v} time (s)"] = r["secs"]
        row[f"{v} chars"]    = len(r["latex"])
        if v == "v3.1":
            row["v3.1 PDF"] = "✅" if r["pdf"] else "❌"
    rows.append(row)

if rows:
    print("\n\n📊 SUMMARY")
    display(pd.DataFrame(rows))
else:
    print("No pages found — upload images in Step 7 first.")

## Step 9 — Download all results 💾

Bundles every generated `.tex` and `.pdf` (samples + your pages) into one zip
and downloads it to your computer.

In [ ]:
import shutil
from google.colab import files as colab_files

n = len(list(RESULTS_DIR.glob("*")))
if n:
    shutil.make_archive("/content/glm_ocr_v31_results", "zip", RESULTS_DIR)
    print(f"📦 {n} files zipped")
    colab_files.download("/content/glm_ocr_v31_results.zip")
else:
    print("No results yet — run Step 6 or Step 8 first.")

---
## ✅ Troubleshooting

| Problem | Fix |
|---|---|
| *No GPU detected* | Runtime → Change runtime type → **T4 GPU** → Save → re-run |
| *CUDA out of memory* | Runtime → Restart session → Run all again |
| *Drive download fails* | Upload the zip manually — see the note in Step 3 |
| *PDF fails to compile* | You still get the `.tex` source — compile-rate is one of the reported benchmark metrics |
| *Session disconnected* | Free Colab idles out after ~90 min; Runtime → Run all to restore |

## ℹ️ About this model

| | |
|---|---|
| **Base model** | [zai-org/GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) (frozen during fine-tuning) |
| **Fine-tuning method** | LoRA (r=32, α=64, dropout=0.05) on the language decoder's q/k/v/o/gate/up/down projections |
| **Training data** | ~4,672 pages of handwritten undergraduate mathematics, annotated by a commercial teacher VLM accessed through an auto-routing API |
| **Training** | Warm-started from the v3 adapter; 2 epochs, 1,168 steps, lr=2e-5, best val loss 0.108 |
| **Adapter size** | ~83 MB |
| **Decoding** | Greedy (deterministic), `repetition_penalty=1.0`, max 2,048 tokens — identical for base and v3.1, so the comparison is fair |
| **Held-out test set** | 700 unseen pages (same benchmark set used for v4.1); v3.1 lifts compile rate from ~0% → 88.9% and CER from 0.515 → 0.397 vs the base model |

*Notebook prepared by Gaurav Vyas (gaurav.vyas.1729@gmail.com).*